# Cleaning — Reddit Comments

Parses raw Reddit comment JSONL files from Bronze layer, computes VADER sentiment per comment, and saves a combined Silver-layer parquet file.

Source:  (structured pipeline section)

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()


files = {
    "conservative": "../../data/1_bronze/reddit/r_conservative_comments.jsonl",
    "worldnews": "../../data/1_bronze/reddit/r_worldnews_comments.jsonl",
    "democrats": "../../data/1_bronze/reddit/r_democrats_comments.jsonl",
    "republican": "../../data/1_bronze/reddit/r_trump_comments.jsonl",
}

## 2. Helper Functions

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # links weg
    text = re.sub(r"\s+", " ", text).strip()     # extra spaties weg
    return text


def parse_comment_file(file_path, subreddit_name):
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            body = obj.get("body", "")
            if not isinstance(body, str):
                continue

            if body.lower() in ["[deleted]", "[removed]"]:
                continue

            created_utc = obj.get("created_utc")
            if created_utc is None:
                continue

            score = obj.get("score", 0)
            author = obj.get("author", None)

            body_clean = clean_text(body)
            if not body_clean:
                continue

            sentiment = sia.polarity_scores(body_clean)["compound"]

            rows.append({
                "subreddit": subreddit_name,
                "created_utc": created_utc,
                "datetime": pd.to_datetime(created_utc, unit="s", utc=True),
                "date": pd.to_datetime(created_utc, unit="s", utc=True).date(),
                "author": author,
                "score": score,
                "body": body,
                "body_clean": body_clean,
                "length_chars": len(body_clean),
                "length_words": len(body_clean.split()),
                "sentiment": sentiment,
                "mentions_trump": int("trump" in body_clean),
                "mentions_harris": int(("harris" in body_clean) or ("kamala" in body_clean)),
                "mentions_biden": int("biden" in body_clean),
            })

    return rows

## 3. Combine all

In [ ]:
all_rows = []

for subreddit, file_path in files.items():
    print(f"Inladen: {subreddit}")
    rows = parse_comment_file(file_path, subreddit)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

print(df.shape)
print(df.head())

In [ ]:
from pathlib import Path
Path("../../data/2_silver/reddit").mkdir(parents=True, exist_ok=True)
df.to_parquet("../../data/2_silver/reddit/reddit_full.parquet", index=False)
print(f"Saved {len(df):,} comments to data/2_silver/reddit/reddit_full.parquet")